In [1]:
import json
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image
from openai import OpenAI
import openai
import os
from pprint import pprint
pd.set_option('display.max_columns', None)

In [2]:
# initialize openai
from dotenv import load_dotenv
load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

In [3]:
import base64
import requests

def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

/Users/igyeongseob/Develop/ai/RAG/vectorDB/vector/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


- 구체적일 수록 좋음
- 하지 말아야 할 것 보다는 해야 할 것 위주로 설명

In [4]:
text_desc = "convert the coat into a street styled fashion."

In [7]:
# GPT를 이용해서 이미지를 읽어와서 description을 생성한다

image_desc_prompt = """
Analyze the user provided input to create a 'modified description' of the image that reflects the 'user input'.

Incorporating user input, the fashion description should only change slightly from the original image description.
The overall color or the details should be the same unless the user input specifically mention modifications.

The item type should only change if the user input specifically mention modifications.
As an example, if the bomber jacket is in the image, the output should also be a bomber jacket unless the user specifies it to change.

Remember, the total length of your response, including all characters and spaces, must stay within the 500-letter constraint. 
Aim for clarity and brevity in your answer.

#user input : {}
""".format(text_desc)

# Path to your image
image_path = "test_images/test_image5.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {openai.api_key}"
}

payload = {
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": image_desc_prompt
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
          }
        }
      ]
    }
  ],
  "max_tokens": 800
}

response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)


In [13]:
img_desc = response.json()['choices'][0]['message']['content']
pprint(img_desc)

('The image features a stylish street fashion look with an edgy oversized pink '
 'coat layered over a chic black turtleneck. A patterned scarf is casually '
 'draped around the neck, complementing the modern vibe. The outfit is paired '
 'with form-fitting black leggings and chunky black ankle boots, enhancing the '
 'urban aesthetic. The backdrop of archways adds a dynamic touch, creating a '
 'perfect setting for this contemporary ensemble. Overall, the look merges '
 'comfort with style, perfect for a street-chic statement.')


dall-e input

In [14]:
text_prompt = """Create a visual representation of the following fashion description. 
Focus on capturing the essence of the outfit in a realistic manner without overcomplication.
The background should be subtle and not detract from the outfit itself, 
Choose a minimalistic background that complements the style of the attire:
The image should be hyper-realistic

Fashion Description:
{}
""".format(img_desc)

In [15]:
pprint(text_prompt)

('Create a visual representation of the following fashion description. \n'
 'Focus on capturing the essence of the outfit in a realistic manner without '
 'overcomplication.\n'
 'The background should be subtle and not detract from the outfit itself, \n'
 'Choose a minimalistic background that complements the style of the attire:\n'
 'The image should be hyper-realistic\n'
 '\n'
 'Fashion Description:\n'
 'The image features a stylish street fashion look with an edgy oversized pink '
 'coat layered over a chic black turtleneck. A patterned scarf is casually '
 'draped around the neck, complementing the modern vibe. The outfit is paired '
 'with form-fitting black leggings and chunky black ankle boots, enhancing the '
 'urban aesthetic. The backdrop of archways adds a dynamic touch, creating a '
 'perfect setting for this contemporary ensemble. Overall, the look merges '
 'comfort with style, perfect for a street-chic statement.\n')


In [ ]:
client = OpenAI()

# "dall-e-3"는 2026-05-12에 API에서 제거(shutdown) 예정
response = client.images.generate(
  model="gpt-image-1-mini",     
  prompt=text_prompt,
  size="1024x1024",
  quality="medium",
  n=1,
)

image_url = response.data[0].b64_json
# url is live for 60 seconds after generation

with open("out.png", "wb") as f:
    f.write(base64.b64decode(image_url))

    

In [20]:
from PIL import Image
from image_utils import draw_images

draw_images(Image.open('oput.png'))

KeyboardInterrupt: 

## Function

In [ ]:
def create_modifications(text_desc, image_path):
    # GPT를 이용해서 이미지를 바탕으로 modified description을 생성

    image_desc_prompt = """
    Analyze the user provided input and ensure the description accurately reflects the 'user preference'.

    Incorporating user input, the modified fashion description emphasizes their unique color palette, 
    showcases an updated fashion style with innovative textiles, introduces intricate patterns for visual interest, 
    and highlights a distinctive shape that redefines their overall silhouette.
    It should consider both the fashion in the image, and the 'user input'.

    Remember, the total length of your response, including all characters and spaces, must stay within the 500-letter constraint. 
    Aim for clarity and brevity in your answer.

    #user input : {}
    """.format(text_desc)

    # # Path to your image
    # image_path = "test_images/test_image5.jpg"

    # Getting the base64 string
    base64_image = encode_image(image_path)

    headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {openai.api_key}"
    }

    payload = {
    "model": "gpt-4o-mini",
    "messages": [
        {
        "role": "user",
        "content": [
            {
            "type": "text",
            "text": image_desc_prompt
            },
            {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
            }
            }
        ]
        }
    ],
    "max_tokens": 800
    }

    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

    # Dall-E intpu
    
    img_desc = response.json()['choices'][0]['message']['content']
    text_desc = "more formal, suitable for a wedding event. a green coat"

    text_prompt = """Create a visual representation of the following fashion description. 
    Focus on capturing the essence of the outfit in a realistic manner without overcomplication.
    The background should be subtle and not detract from the outfit itself, 
    Choose a minimalistic background that complements the style of the attire:

    Fashion Description:
    {}
    """.format(img_desc)
    

    client = OpenAI()

    response = client.images.generate(
        model="gpt-image-1-mini",
        prompt=text_prompt + "\nStyle: vivid, high saturation, cinematic lighting",
        size="1024x1024",
        quality="medium",  # low|medium|high|auto
        n=1,
    )   

    img_b64 = response.data[0].b64_json
    # url is live for 60 seconds after generation
    return base64.b64decode(img_b64)